<img src="https://geo-credito-rural.github.io/_static/logo.jpeg" align="right" width="64" />

# <span style="color:#336699">Impedimentos Sociais, Ambientais e Climáticos</span>

<hr style="border:2px solid #0077b9;">
<div style="text-align: left;">
    <a href="https://nbviewer.jupyter.org/github/brazil-data-cube/code-gallery/blob/master/jupyter/Python/stac/stac-image-processing.ipynb"><img src="https://raw.githubusercontent.com/jupyter/design/master/logos/Badges/nbviewer_badge.svg" align="center"/></a>
</div>

<br/>

<div style="text-align: center;font-size: 90%;">
     Gabriel Sansigolo, Thales Sehn Körting, Gilberto Queiroz, Karine Ferreira, Marcos Adami
    <br/><br/>
    Divisão de Observação da Terra e Geoinformática, Instituto Nacional de Pesquisas Espaciais (INPE)
    <br/>
    Avenida dos Astronautas, 1758, Jardim da Granja, São José dos Campos, SP 12227-010, Brazil
    <br/><br/>
    Contato: <a href="https://geo-credito-rural.github.io/">Equipe - Geo Credito Rural</a>
    <br/><br/>
    Última atualização: 11 de abril de 2026
</div>


<br/>

</div>

Esse notebook aborda de forma simplificada as restrições legais para o acesso ao crédito rural com base em critérios sociais, ambientais e climáticos.

# <span style="color:#336699">Contextualização</span>
<hr style="border:1px solid #0077b9;">

Com as recentes atualizações do Conselho Monetário Nacional (como as Resoluções CMN Nº 5.193/2024, 5.267/2025 e 5.268/2025), a verificação de financiamentos tornou-se mais rigorosa. Hoje, exige-se a identificação da propriedade via Cadastro Ambiental Rural (CAR) e o monitoramento obrigatório por sensoriamento remoto para áreas superiores a 300 hectares, garantindo que os recursos financiados respeitem a conformidade socioambiental.

Na atividade prática de hoje, vamos analisar os impedimentos definidos para esses empreendimentos.

# <span style="color:#336699">4 - Terras Indígenas</span>
<hr style="border:1px solid #0077b9;">

Terra Indígena, conforme a Constituição Federal de 1988, é um território demarcado e protegido para a posse permanente e o usufruto exclusivo dos povos indígenas. Essas terras são reconhecidas como patrimônio da União e são destinadas à preservação de sua cultura, tradições, recursos naturais e formas de organização social, além de assegurar a reprodução física e cultural dessas comunidades. A demarcação das terras indígenas é um direito constitucional e visa garantir a autodeterminação, a autonomia e a proteção dos direitos dos povos indígenas, bem como sua participação ativa na gestão e preservação desses territórios.

De acordo com a Resolução CMN Nº 5.193/2024:

> 7 - Não será concedido crédito rural para empreendimento situado em imóvel rural total ou parcialmente inserido em
terras ocupadas por indígenas, observado que: (Res CMN 5.193 art 1º)
a) as terras ocupadas por indígenas devem constar como homologadas, regularizadas ou definidas como Reserva
Indígena no Sistema Indigenista de Informações da Fundação Nacional dos Povos Indígenas (Funai); e
b) o disposto no caput não se aplica aos casos em que o proponente pertença aos povos ou às comunidades
indígenas ocupantes ou habitantes da terra indígena na qual se situa o empreendimento.

Uma representação ilustrativa em forma de diagrama da restrição pode ser observada na Figura 4:

![car](https://i.imgur.com/SecW4rk.png "Terras Indígenas")

Prática: vamos simular a verificação de sobreposição.




**Configuração e Exemplo Não atende a norma**

In [ ]:
from shapely.geometry import Polygon
import matplotlib.pyplot as plt
import requests, zipfile, io
import geopandas as gpd

In [ ]:
empreendimento = {
    "geometria": Polygon([(0, 0), (1, 0), (1, 1), (0, 1)]),
    "comunidade_pertencente": "Guarani",
    "proponente": "João Guarani"
}

In [ ]:
terras_indigenas = [
    {"geometria": Polygon([(0.5, 0.5), (1.5, 0.5), (1.5, 1.5), (0.5, 1.5)]), "homologada_funai": True, "comunidade_habitante": "Guarani"},
    {"geometria": Polygon([(5, 5), (6, 5), (6, 6), (5, 6)]), "homologada_funai": False, "comunidade_habitante": "Tupinambá"},
    {"geometria": Polygon([(10, 10), (11, 10), (11, 11), (10, 11)]), "homologada_funai": True, "comunidade_habitante": "Xavante"}
]

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))

x_emp, y_emp = empreendimento["geometria"].exterior.xy

ax.plot(x_emp, y_emp, color='#1f77b4', linewidth=2, label='Empreendimento')
ax.fill(x_emp, y_emp, color='#1f77b4', alpha=0.3)

for i, ti in enumerate(terras_indigenas):
    x_ti, y_ti = ti["geometria"].exterior.xy

    label = 'Terra Indígena' if i == 0 else None

    ax.plot(x_ti, y_ti, color='#ff7f0e', linewidth=2, label=label)
    ax.fill(x_ti, y_ti, color='#ff7f0e', alpha=0.3)

ax.set_aspect('equal')
ax.set_title('Visualização: Empreendimento vs. Terras Indígenas')
ax.set_xlabel('Longitude / X')
ax.set_ylabel('Latitude / Y')
ax.legend()
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

In [ ]:
imovel_atende_norma = True
motivo_impedimento = ""

for i, terra in enumerate(terras_indigenas):

    # Verifica se a geometria do empreendimento possui sobreposição com a terra indígena atual.
    if empreendimento["geometria"].intersects(terra["geometria"]):

        # Verifica se a terra indígena consta como homologada (regularizada ou reserva) na Funai.
        if terra["homologada_funai"]:

            # Se for homologada pela Funai, o empreendimento não pode estar lá, portanto não atende a norma.
            imovel_atende_norma = False
            motivo_impedimento = f"sobreposição com terra indígena homologada (Item {i})."
            break

        # Caso a terra não seja homologada, verifica se o proponente pertence à comunidade habitante daquela terra.
        else:

            # Compara se o nome da comunidade no empreendimento é o mesmo da lista da terra indígena.
            if empreendimento["comunidade_pertencente"] == terra["comunidade_habitante"]:
                # Se pertence à comunidade habitante, atende a norma para esta área.
                pass
            else:
                # Se não pertence à comunidade da área em disputa, não atende a norma.
                imovel_atende_norma = False
                motivo_impedimento = f"proponente não pertence à comunidade habitante da terra não homologada (Item {i})."
                break

    # Caso não exista sobreposição com esta terra indígena específica.
    else:
        pass

# Verifica se o empreendimento passou por todas as checagens sem ser marcado com impedimento.
if imovel_atende_norma:
    print("O empreendimento atende a norma.")
# Caso tenha falhado em alguma das condições acima.
else:
    print(f"O empreendimento não atende a norma porque houve: {motivo_impedimento}")

**Configuração e Exemplo atende a norma**

In [ ]:
empreendimento = {
    "geometria": Polygon([(0, 0), (1, 0), (1, 1), (0, 1)]),
    "comunidade_pertencente": "Guarani",
    "proponente": "João Guarani"
}

In [ ]:
terras_indigenas = [
    {"geometria": Polygon([(0.5, 0.5), (1.5, 0.5), (1.5, 1.5), (0.5, 1.5)]), "homologada_funai": False, "comunidade_habitante": "Guarani"},
    {"geometria": Polygon([(5, 5), (6, 5), (6, 6), (5, 6)]), "homologada_funai": True, "comunidade_habitante": "Tupinambá"},
    {"geometria": Polygon([(10, 10), (11, 10), (11, 11), (10, 11)]), "homologada_funai": True, "comunidade_habitante": "Xavante"}
]

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))

x_emp, y_emp = empreendimento["geometria"].exterior.xy

ax.plot(x_emp, y_emp, color='#1f77b4', linewidth=2, label='Empreendimento')
ax.fill(x_emp, y_emp, color='#1f77b4', alpha=0.3)

for i, ti in enumerate(terras_indigenas):
    x_ti, y_ti = ti["geometria"].exterior.xy

    label = 'Terra Indígena' if i == 0 else None

    ax.plot(x_ti, y_ti, color='#ff7f0e', linewidth=2, label=label)
    ax.fill(x_ti, y_ti, color='#ff7f0e', alpha=0.3)

ax.set_aspect('equal')
ax.set_title('Visualização: Empreendimento vs. Terras Indígenas')
ax.set_xlabel('Longitude / X')
ax.set_ylabel('Latitude / Y')
ax.legend()
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

In [ ]:
imovel_atende_norma = True
motivo_impedimento = ""

for i, terra in enumerate(terras_indigenas):

    # Verifica se a geometria do empreendimento possui sobreposição com a terra indígena atual.
    if empreendimento["geometria"].intersects(terra["geometria"]):

        # Verifica se a terra indígena consta como homologada (regularizada ou reserva) na Funai.
        if terra["homologada_funai"]:

            # Se for homologada pela Funai, o empreendimento não pode estar lá, portanto não atende a norma.
            imovel_atende_norma = False
            motivo_impedimento = f"sobreposição com terra indígena homologada (Item {i})."
            break

        # Caso a terra não seja homologada, verifica se o proponente pertence à comunidade habitante daquela terra.
        else:

            # Compara se o nome da comunidade no empreendimento é o mesmo da lista da terra indígena.
            if empreendimento["comunidade_pertencente"] == terra["comunidade_habitante"]:
                # Se pertence à comunidade habitante, atende a norma para esta área.
                pass
            else:
                # Se não pertence à comunidade da área em disputa, não atende a norma.
                imovel_atende_norma = False
                motivo_impedimento = f"proponente não pertence à comunidade habitante da terra não homologada (Item {i})."
                break

    # Caso não exista sobreposição com esta terra indígena específica.
    else:
        pass

# Verifica se o empreendimento passou por todas as checagens sem ser marcado com impedimento.
if imovel_atende_norma:
    print("O empreendimento atende a norma.")
# Caso tenha falhado em alguma das condições acima.
else:
    print(f"O empreendimento não atende a norma porque houve: {motivo_impedimento}")

**Exemplo com dados reais**

**Configuração e Exemplo Não atende a norma**

In [ ]:
r = requests.get("https://raw.githubusercontent.com/GSansigolo/shapefiles/refs/heads/main/mini_tis_ma.zip")

zipfile.ZipFile(io.BytesIO(r.content)).extractall("./mini_tis_ma.zip")
file_path = "./mini_tis_ma.zip/mini_tis_ma.shp"

mini_tis = gpd.read_file(file_path)

In [ ]:
r = requests.get("https://raw.githubusercontent.com/GSansigolo/shapefiles/refs/heads/main/poligono_3.zip")

zipfile.ZipFile(io.BytesIO(r.content)).extractall("./poligono_3.zip")
file_path = "./poligono_3.zip/poligono_3.shp"

poligono_3 = gpd.read_file(file_path)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, ax = plt.subplots(figsize=(8, 8))

mini_tis.plot(ax=ax, color='#d62728', alpha=0.3)
mini_tis.boundary.plot(ax=ax, color='#d62728', linewidth=2)

poligono_3.plot(ax=ax, color='#1f77b4', alpha=0.3)
poligono_3.boundary.plot(ax=ax, color='#1f77b4', linewidth=2)

ax.set_aspect('equal')
ax.set_title('Visualização: Empreendimento vs. Lista Terra Indígena')
ax.set_xlabel('Longitude / X')
ax.set_ylabel('Latitude / Y')
plt.grid(True, linestyle='--', alpha=0.6)

legenda_empreendimento = mpatches.Patch(color='#1f77b4', alpha=0.5, label='Empreendimento')
legenda_tis = mpatches.Patch(color='#d62728', alpha=0.5, label='Terra Indígena')
ax.legend(handles=[legenda_empreendimento, legenda_tis])
plt.show()

In [ ]:
poligono_3 = poligono_3.to_crs(mini_tis.crs)
geom_empreendimento = poligono_3.geometry.iloc[0]

mini_tis_plano = mini_tis.to_crs(epsg=6933)
geom_empreendimento_plana = poligono_3.to_crs(epsg=6933).geometry.iloc[0]

sobreposicoes = mini_tis.intersects(geom_empreendimento)
areas = mini_tis_plano.intersection(geom_empreendimento_plana).area

area_minima = 1.0

for i, sobrepoe in sobreposicoes.items():
    if sobrepoe and areas[i] > area_minima:
        print(f"O empreendimento sobrepõe a área da Terra Indígena {i}! Área da sobreposição: {areas[i]:.2f} m².")


**Configuração e Exemplo atende a norma**

In [ ]:
r = requests.get("https://raw.githubusercontent.com/GSansigolo/shapefiles/refs/heads/main/poligono_4.zip")

zipfile.ZipFile(io.BytesIO(r.content)).extractall("./poligono_4.zip")
file_path = "./poligono_4.zip/poligono_4.shp"

poligono_4 = gpd.read_file(file_path)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, ax = plt.subplots(figsize=(8, 8))

mini_tis.plot(ax=ax, color='#d62728', alpha=0.3)
mini_tis.boundary.plot(ax=ax, color='#d62728', linewidth=2)

poligono_4.plot(ax=ax, color='#1f77b4', alpha=0.3)
poligono_4.boundary.plot(ax=ax, color='#1f77b4', linewidth=2)

ax.set_aspect('equal')
ax.set_title('Visualização: Empreendimento vs. Lista Terra Indígena')
ax.set_xlabel('Longitude / X')
ax.set_ylabel('Latitude / Y')
plt.grid(True, linestyle='--', alpha=0.6)

legenda_empreendimento = mpatches.Patch(color='#1f77b4', alpha=0.5, label='Empreendimento')
legenda_tis = mpatches.Patch(color='#d62728', alpha=0.5, label='Terra Indígena')
ax.legend(handles=[legenda_empreendimento, legenda_tis])
plt.show()

In [ ]:
poligono_4 = poligono_4.to_crs(mini_tis.crs)
geom_empreendimento = poligono_4.geometry.iloc[0]

mini_tis_plano = mini_tis.to_crs(epsg=6933)
geom_empreendimento_plana = poligono_4.to_crs(epsg=6933).geometry.iloc[0]

sobreposicoes = mini_tis.intersects(geom_empreendimento)
areas = mini_tis_plano.intersection(geom_empreendimento_plana).area

area_minima = 1.0

for i, sobrepoe in sobreposicoes.items():
    if sobrepoe and areas[i] > area_minima:
        print(f"O empreendimento sobrepõe a área da Terra Indígena {i}! Área da sobreposição: {areas[i]:.2f} m².")
